In [21]:
import pandas as pd
import os
import re

def list_video_files(full_folder_path):
    files = os.listdir(full_folder_path)
    full_paths = [os.path.join(full_folder_path, f) for f in files if f.endswith('.mp4')]
    return full_paths

def find(image_id, files):
    pattern = re.compile(rf"{re.escape(image_id)}_\d+\.png$")
    for f in files:
        if pattern.search(f):
            return f
    return None


# Read the CSV file (edit the path if needed)
csv_path = fr"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound.csv"
df = pd.read_csv(csv_path, header=None, names=["youtube_id", "start_second", "label", "split"])




In [22]:
import os
import cv2
import torch
from PIL import Image
import re
from transformers import CLIPProcessor, CLIPModel

import os
import re
import torch

from moviepy.editor import VideoFileClip


device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP and processor from Hugging Face
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def process_video_and_save_frame(video_path, caption, save_dir):
    base = os.path.basename(video_path)
    match = re.match(r"(.+?)_(\d+)\.mp4$", base)
    if not match:
        print(f"Failed to extract ID and start second from {video_path}")
        return
    youtube_id, start_second = match.group(1), int(match.group(2))
    clean_id = youtube_id.lstrip('-')

    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Failed to open video: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_interval = max(1, int(fps / 10))  # Ensure frame_interval is not zero


    # Process text once
    inputs_text = processor(text=[caption], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_embeds = model.get_text_features(**inputs_text)
        text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)

    max_similarity = -1
    best_frame_img = None
    best_preprocessed_tensor = None
    similarity_treshold = 0.25

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_interval == 0:
            image_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            inputs_image = processor(images=image_pil, return_tensors="pt").to(device)

            with torch.no_grad():
                image_embeds = model.get_image_features(**inputs_image)
                image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)

            similarity = (text_embeds @ image_embeds.T).item()
            if similarity > max_similarity:
                max_similarity = similarity
                best_frame_img = image_pil
                best_preprocessed_tensor = inputs_image['pixel_values'].cpu()

        frame_count += 1

    cap.release()

    if best_frame_img and max_similarity > similarity_treshold:
        image_save_path = os.path.join(save_dir, f"{clean_id}_{start_second}.png")
        # tensor_save_path = os.path.join(save_dir, f"{clean_id}_{start_second}_preprocessed.pt")
        best_frame_img.save(image_save_path)
        # torch.save(best_preprocessed_tensor, tensor_save_path)
        print(f"Saved best matching frame to {image_save_path} with similarity {max_similarity:.4f}")
        # print(f"Saved preprocessed tensor to {tensor_save_path}")
    else:
        print(f"No frames processed for video: {video_path}")
        print(f"Max similarity {max_similarity:.4f} did not exceed threshold {similarity_treshold}")

    return max_similarity > similarity_treshold


def save_full_audio_moviepy(video_path, save_dir):
    base = os.path.basename(video_path)
    filename, _ = os.path.splitext(base)
    clean_id = filename.lstrip('-')
    
    os.makedirs(save_dir, exist_ok=True)

    try:
        video_clip = VideoFileClip(video_path)
        audio = video_clip.audio
        audio_save_path = os.path.join(save_dir, f"{clean_id}.wav")
        audio.write_audiofile(audio_save_path)
        print(f"Saved full audio to {audio_save_path}")
    except Exception as e:
        print(f"Failed to extract audio from {video_path}, error: {e}")


In [23]:
def extract_youtube_id_and_start_second(filename):
    """
    Extracts YouTube ID and start second from a filename like '-_BH-TPpLWk_000035.mp4'
    """
    base = os.path.basename(filename)
    match = re.match(r"(.+?)_(\d+)\.mp4$", base)
    if match:
        return match.group(1), int(match.group(2))
    else:
        return None, None

In [ ]:
# Example usage
base_paths = [
        r"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_04\train_data\VGGSound_final",

    ]

for base_path in base_paths:

    video_dir_path = os.path.join(base_path, "video")
    audio_dir_path = os.path.join(base_path, "audio")
    image_dir_path = os.path.join(base_path, "image")

    video_file_paths = list_video_files(video_dir_path)

    print("Video directory path:", video_dir_path)
    print("Audio directory path:", audio_dir_path)
    print("Image directory path:", image_dir_path)


    # Iterate through all video files
    for path in video_file_paths:
        youtube_id, file_start_second = extract_youtube_id_and_start_second(path)
        if youtube_id and file_start_second is not None:
            clean_id = youtube_id.lstrip('-')  # For display
            # Match CSV rows by ID and exact start_second
            matches = df[(df['youtube_id'] == youtube_id) & (df['start_second'] == file_start_second)]
            if not matches.empty:
                for _, row in matches.iterrows():
                    caption = row['label']
                    print(f"Processing {clean_id}_{file_start_second}: {caption} | Path: {path}")
                    # Call the frame extraction function with video path, caption, and save directory
                    status = process_video_and_save_frame(
                        video_path=path,
                        caption=caption,
                        save_dir=image_dir_path
                    )

                    if status:
                        save_full_audio_moviepy(path, audio_dir_path)
                    
                    
            else:
                print(f"No exact match for {clean_id}_{file_start_second} found in CSV. ❌ | Path: {path}")
        else:
            print(f"Failed to extract ID or start second from filename: {path}")


Video directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\video
Audio directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\audio
Image directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\image


KeyboardInterrupt: 

In [28]:
# Example usage
base_paths = {
        "vggsound_00" : r"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final",
        "vggsound_02" : r"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_02\scratch\shared\beegfs\hchen\train_data\VGGSound_final",
        "vggsound_04" : r"D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_04\train_data\VGGSound_final",


}

# Create an empty DataFrame with specified columns
new_df = pd.DataFrame(columns=['base_folder', 'image_file', 'audio_file', 'caption'])


for base_folder in base_paths:
    print(f"Processing base folder: {base_folder}")
    base_path = base_paths[base_folder]

    video_dir_path = os.path.join(base_path, "video")
    audio_dir_path = os.path.join(base_path, "audio")
    image_dir_path = os.path.join(base_path, "image")

    video_file_paths = list_video_files(video_dir_path)

    print("Video directory path:", video_dir_path)
    print("Audio directory path:", audio_dir_path)
    print("Image directory path:", image_dir_path)


    # Iterate through all video files
    for path in video_file_paths:
        youtube_id, file_start_second = extract_youtube_id_and_start_second(path)
        
        if youtube_id and file_start_second is not None:
            clean_id = youtube_id.lstrip('-')  # For display
            
            image_file = find(clean_id, os.listdir(image_dir_path))
            if image_file:

                audio_file = image_file.replace('.png', '')

                # audio_file currently like "5pcv3sZHEE_3"
                m = re.search(r"_(\d+)$", audio_file)
                if m:
                    stime = m.group(1).zfill(6)
                else:
                    stime = audio_file.split("_")[-1].zfill(6)

                audio_file = f"{clean_id}_{stime}.wav"

                if audio_file in os.listdir(audio_dir_path):
                    # # Match CSV rows by ID and exact start_second
                    matches = df[(df['youtube_id'] == youtube_id) & (df['start_second'] == file_start_second)]
                    if not matches.empty:
                        for _, row in matches.iterrows():
                            caption = row['label']
                    
                            # print(base_folder, image_file, audio_file, caption)
                            # Append a new row using loc
                            new_df.loc[len(new_df)] = [base_folder, image_file, audio_file, caption]

                    pass
                else:
                    print(f"Audio {audio_file} not found in {audio_dir_path}")

        else:
            print(f"Failed to extract ID or start second from filename: {path}")



# new_df.to_csv('output.csv', index=False, encoding='utf-8')


Processing base folder: vggsound_00
Video directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\video
Audio directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\audio
Image directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_00\train_data\VGGSound_final\image
Processing base folder: vggsound_02
Video directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_02\scratch\shared\beegfs\hchen\train_data\VGGSound_final\video
Audio directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_02\scratch\shared\beegfs\hchen\train_data\VGGSound_final\audio
Image directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_02\scratch\shared\beegfs\hchen\train_data\VGGSound_final\image
Processing base folder: vggsound_04
Video directory path: D:\SIT\Uni\SIT\Y2T1\CVDL\Phase2\VGG-Sound\vggsound\vggsound_04\train_data\VGGSound_f

In [29]:
new_df

,base_folder,image_file,audio_file,caption
0,vggsound_00,g-f_I2yQ_1.png,g-f_I2yQ_000001.wav,people marching
1,vggsound_00,0PQM4-hqg_30.png,0PQM4-hqg_000030.wav,waterfall burbling
2,vggsound_00,56QUhyDQM_185.png,56QUhyDQM_000185.wav,playing tennis
3,vggsound_00,5OkAjCI7g_40.png,5OkAjCI7g_000040.wav,people belly laughing
4,vggsound_00,8puiAGLhs_30.png,8puiAGLhs_000030.wav,car engine starting
...,...,...,...,...
22534,vggsound_04,BdNrtTBYFME_249.png,BdNrtTBYFME_000249.wav,church bell ringing
22535,vggsound_04,BdoxJxLQSIE_13.png,BdoxJxLQSIE_000013.wav,ice cracking
22536,vggsound_04,BdoxJxLQSIE_13.png,BdoxJxLQSIE_000013.wav,ice cracking
22537,vggsound_04,BDOxmZ-2O88_30.png,BDOxmZ-2O88_000030.wav,"motorboat, speedboat acceleration"


In [30]:
print("DataFrame before removing duplicates:", new_df.shape)


DataFrame before removing duplicates: (22539, 4)


In [34]:

# Remove duplicates considering all 4 columns
updated_df = new_df.drop_duplicates(subset=['base_folder', 'image_file', 'audio_file', 'caption'])

print("DataFrame after removing duplicates:", updated_df.shape)

# Save to CSV
updated_df.to_csv('main_dataV1.csv', index=False, encoding='utf-8')


DataFrame after removing duplicates: (18643, 4)
